# Snippet from Math-Lyapunov-Stability.md


In [ ]:
import numpy as np
from typing import Tuple, Optional

class LyapunovController:
    """
    SRMF Controller with Lyapunov-based gating.
    
    Implements discrete Lyapunov stability checks to ensure
    non-increasing free energy along trajectories.
    """
    
    def __init__(
        self,
        initial_radius: float = 1.0,
        shrink_factor: float = 0.5,
        expand_factor: float = 1.1,
        ema_alpha: float = 0.2,
        threshold: float = 0.0
    ):
        """
        Initialize Lyapunov controller.
        
        Args:
            initial_radius: Starting trust radius.
            shrink_factor: Multiplier on violation (beta).
            expand_factor: Multiplier on success (alpha).
            ema_alpha: Exponential moving average weight (lambda).
            threshold: Tolerance for delta-V acceptance.
        """
        self.r_t = initial_radius
        self.beta = shrink_factor
        self.alpha = expand_factor
        self.lambda_ema = ema_alpha
        self.threshold = threshold
        self.delta_ema = 0.0
        self.history = []
    
    def lyapunov_gate(
        self, 
        E_t: float, 
        E_tp1: float
    ) -> Tuple[float, bool, dict]:
        """
        Gate update based on Lyapunov criterion.
        
        Args:
            E_t: Current free energy (Lyapunov function value).
            E_tp1: Proposed free energy after update.
            
        Returns:
            Tuple of (new_radius, accept_flag, metrics_dict).
        """
        delta_V = E_tp1 - E_t
        
        # Update EMA for smoothed drift signal
        self.delta_ema = (
            self.lambda_ema * delta_V + 
            (1 - self.lambda_ema) * self.delta_ema
        )
        
        # Gate decision
        if delta_V > self.threshold:
            # Violation: energy increase detected
            r_new = self.r_t * self.beta
            accept = False
            status = "REJECT"
        else:
            # Safe: energy decrease or stable
            r_new = min(self.r_t * self.alpha, 2.0)  # Cap expansion at 2x initial
            accept = True
            status = "ACCEPT"
        
        # Update trust radius with floor
        self.r_t = max(r_new, 0.01)  # Floor at 1% of initial
        
        # Log metrics
        metrics = {
            "delta_V": delta_V,
            "delta_ema": self.delta_ema,
            "radius": self.r_t,
            "status": status,
            "threshold": self.threshold
        }
        self.history.append(metrics)
        
        return self.r_t, accept, metrics

# Example: Trajectory Validation
def validate_trajectory():
    """Demonstrate Lyapunov gating on a noisy descent trajectory."""
    
    # Simulated free energy trajectory with noise
    E_trajectory = [
        1.00,  # Initial state
        0.75,  # Good descent
        0.60,  # Continued descent
        0.65,  # Small increase (noise violation)
        0.50,  # Recovery descent
        0.55,  # Another violation
        0.40,  # Strong descent
        0.35   # Convergence
    ]
    
    controller = LyapunovController(
        initial_radius=1.0,
        shrink_factor=0.5,
        expand_factor=1.05,
        ema_alpha=0.3
    )
    
    print("=" * 60)
    print("Lyapunov Trajectory Validation")
    print("=" * 60)
    
    for t in range(1, len(E_trajectory)):
        E_t = E_trajectory[t-1]
        E_tp1 = E_trajectory[t]
        
        r_new, accepted, metrics = controller.lyapunov_gate(E_t, E_tp1)
        
        symbol = "✓" if accepted else "✗"
        print(f"\nStep {t}: {symbol} {metrics['status']}")
        print(f"  E_t={E_t:.3f} → E_t+1={E_tp1:.3f}")
        print(f"  ΔV={metrics['delta_V']:+.3f} (EMA: {metrics['delta_ema']:+.3f})")
        print(f"  Trust Radius: r={metrics['radius']:.3f}")
    
    print("\n" + "=" * 60)
    print("Summary Statistics")
    print("=" * 60)
    
    accepts = sum(1 for h in controller.history if h['status'] == 'ACCEPT')
    rejects = len(controller.history) - accepts
    final_radius = controller.history[-1]['radius']
    
    print(f"Accepted: {accepts}/{len(controller.history)} "
          f"({100*accepts/len(controller.history):.1f}%)")
    print(f"Rejected: {rejects}/{len(controller.history)} "
          f"({100*rejects/len(controller.history):.1f}%)")
    print(f"Final Radius: {final_radius:.3f}")
    print(f"Final EMA Drift: {controller.delta_ema:+.4f}")

if __name__ == "__main__":
    validate_trajectory()
